# **CrewAI Agent with a Custom Tool**

In [3]:
!pip install -q crewai crewai_tools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.8/189.8 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 820.8/820.8 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4

In [4]:
import crewai
import crewai_tools

print(crewai.__version__)
print(crewai_tools.__version__)

1.15.10
1.15.10


# Set API Keys

In [5]:
from google.colab import userdata
import os
# os.environ["SERPER_API_KEY"] = userdata.get('SERPER_API_KEY')
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY_NEW')
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')


# Import Dependencies

In [6]:
from crewai import Agent, Crew, Task, Process, LLM, Process
from crewai_tools import SerperDevTool, DirectoryReadTool
from pydantic import BaseModel
from typing import List

# from dotenv import load_dotenv
# load_dotenv()

#------------------------------------------------------------------------
import warnings
warnings.filterwarnings("ignore")
#------------------------------------------------------------------------


# Define LLMs

In [7]:
# Create an LLM with a temperature of 0 to ensure deterministic outputs
# -----------
# Create LLM
# -----------

# Create an LLM with a temperature of 0 to ensure deterministic outputs
# OPENAI LLMs
# llm = LLM(
#          # model="gpt-5.4-mini",
#           model="gpt-5.4-nano",
#           base_url="https://api.openai.com/v1",
#           api_key = os.environ["OPENAI_API_KEY"],
#           temperature=0.2)

# GROQ hosted LLMs
llm = LLM(
     model="llama-3.3-70b-versatile",
     base_url="https://api.groq.com/openai/v1",
     api_key=os.environ["GROQ_API_KEY"],
     temperature=0.4)



# Define a Custom Tools

In [8]:
from typing import Type
from crewai.tools import BaseTool
from pydantic import BaseModel, Field
#-------------------------------------------------------------------
# Define a custom input data type for the tool
class MyToolInput(BaseModel):
    """Input schema for CustomTool."""
    tool_message: str = Field(..., description="This is a text message.")
    tool_number: int = Field(..., description="This is a number.")
#-------------------------------------------------------------------
# Define the tool
class CustomTool(BaseTool):
    name: str = "Custom Tool to echo message."
    description: str = "This tool echos message. It's vital for echoing messages."
    args_schema: Type[BaseModel] = MyToolInput

    def _run(self, tool_message: str, tool_number: int) -> str:
        # Your tool's logic here
        print("\n Runnnig custom tool......")
        return f"Echoing message: {tool_message}. Long live {tool_number} years + eternity!"


# Define Agents

In [9]:
# Create an agent using the tool

agent = Agent(
    role="Echo Agent",
    goal="Echo back input using custom tool",
    backstory="You are expert in echoing messages",
    tools=[CustomTool()],
    llm=llm,
    verbose=True
)


# Define Tasks

In [10]:
task = Task(
    description="Echo the provided message {message} {number}.",
    expected_output="You need to echo the message returned by the tool.",
    agent=agent,
    verbose=True
)

# Define Crew (Orchestration Layer)

In [11]:
crew = Crew(
    agents=[agent],
    tasks=[task]
    )

# Run the Crew

In [12]:
result = await crew.kickoff_async(inputs={ "message": "Hello, India", "number": "1100"})
# result = await crew.kickoff_async(inputs={ "number": "1100", "message": "Hello, India"})

print(result)


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Echo Agent                                                                                              │
│                                                                                                                 │
│  Task: Echo the provided message Hello, India 1100.                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


 Runnnig custom tool......
Tool custom_tool_to_echo_message executed with result: Echoing message: Hello, India. Long live 1100 years + eternity!...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Echo Agent                                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Hello, India. Long live 1100 years + eternity!                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Hello, India. Long live 1100 years + eternity!


In [13]:
result = await crew.kickoff_async(inputs={ "number": "1100", "message": "Hello, India"})
print(result)


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Echo Agent                                                                                              │
│                                                                                                                 │
│  Task: Echo the provided message Hello, India 1100.                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


 Runnnig custom tool......
Tool custom_tool_to_echo_message executed with result: Echoing message: Hello, India. Long live 1100 years + eternity!...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Echo Agent                                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Hello, India. Long live 1100 years + eternity!                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Hello, India. Long live 1100 years + eternity!


# **Asynchronous Tool**

CrewAI supports asynchronous tools, allowing you to implement tools that perform non-blocking operations like network requests, file I/O, or other async operations without blocking the main execution thread.

## 1. Using the tool Decorator with Async Functions

In [21]:
from crewai.tools import tool
import asyncio

# -----------------------------------------------------
# Define the custom async tool
# -----------------------------------------------------

@tool("fetch_data_async")
async def fetch_data_async(query: str) -> str:
    """Asynchronously fetch data based on the query."""
    # Simulate async operation
    await asyncio.sleep(1)
    return f"Data retrieved for {query}"

In [ ]:

# -----------------------------------------------------
# Configure the LLM
# -----------------------------------------------------
# GROQ hosted LLMs
llm = LLM(
     model="llama-3.3-70b-versatile",
     base_url="https://api.groq.com/openai/v1",
     api_key=os.environ["GROQ_API_KEY"],
     temperature=0.4)


In [26]:
# -----------------------------------------------------
# Define the agent
# -----------------------------------------------------
research_agent = Agent(
    role="Research Assistant",
    goal="Retrieve information using the available tool and summarize it.",
    backstory=(
        "You are an AI research assistant that can fetch external "
        "information using custom tools."
    ),
    tools=[fetch_data_async],

    llm=llm,
    verbose=True
)

# -----------------------------------------------------
# Define the task
# -----------------------------------------------------

research_task = Task(
    description=(
        "Find information about CrewAI asynchronous tools "
        "by using the available tool. "
        "Summarize the retrieved information."
    ),
    expected_output="A short summary of the retrieved data.",
    agent=research_agent
)

# -----------------------------------------------------
# Define the crew
# -----------------------------------------------------

crew = Crew(
    agents=[research_agent],
    tasks=[research_task],
    process=Process.sequential,
    verbose=True
)


In [28]:
# -----------------------------------------------------
# Execute
# -----------------------------------------------------

result = await crew.kickoff_async()

print(result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 56b49adb-614f-4b0b-abec-a0a31ad4bc64                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Find information about CrewAI asynchronous tools by using the available tool. Summarize the retrieved    │
│  information.                                                                                                   │
│  ID: 77d83b26-b18a-4bc6-a5cf-384c8c729748                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Assistant                                                                                      │
│                                                                                                                 │
│  Task: Find information about CrewAI asynchronous tools by using the available tool. Summarize the retrieved    │
│  information.                                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: fetch_data_async                                                                                         │
│  Args: {'query': 'CrewAI asynchronous tools'}                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool fetch_data_async executed with result: Data retrieved for CrewAI asynchronous tools...

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Assistant                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  CrewAI asynchronous tools are designed to streamline workflows and enhance productivity by allowing for        │
│  non-blocking, concurrent execution of tasks. These tools enable developers to write single-threaded code that  │
│  can handle multiple tasks simultaneously, improving overall system performance and responsiveness.             │
│                                                                                                                 │
│  Key features of CrewAI asynchronous tools include:                                                             │
│                                                                                                                 │
│  1. Asynchronous Programming: CrewAI asynchronous tools support asynchronous programming models, enabling       │
│  developers to write code that can execute multiple tasks concurrently without blocking or waiting for each     │
│  task to complete.                                                                                              │
│  2. Non-Blocking I/O: CrewAI asynchronous tools provide non-blocking I/O operations, allowing developers to     │
│  perform input/output operations without blocking the execution of other tasks.                                 │
│  3. Concurrent Execution: CrewAI asynchronous tools enable concurrent execution of tasks, improving system      │
│  performance and responsiveness by utilizing multiple CPU cores and minimizing idle time.                       │
│  4. Task Queuing: CrewAI asynchronous tools often include task queuing mechanisms, which allow developers to    │
│  schedule tasks for execution and manage task priorities.                                                       │
│  5. Error Handling: CrewAI asynchronous tools typically provide robust error handling mechanisms, enabling      │
│  developers to handle errors and exceptions that may occur during asynchronous task execution.                  │
│                                                                                                                 │
│  By leveraging CrewAI asynchronous tools, developers can build scalable, high-performance applications that     │
│  can handle large volumes of concurrent requests and tasks, improving overall system efficiency and user        │
│  experience.                                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Find information about CrewAI asynchronous tools by using the available tool. Summarize the retrieved    │
│  information.                                                                                                   │
│  Agent: Research Assistant                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

CrewAI asynchronous tools are designed to streamline workflows and enhance productivity by allowing for non-blocking, concurrent execution of tasks. These tools enable developers to write single-threaded code that can handle multiple tasks simultaneously, improving overall system performance and responsiveness.

Key features of CrewAI asynchronous tools include:

1. Asynchronous Programming: CrewAI asynchronous tools support asynchronous programming models, enabling developers to write code that can execute multiple tasks concurrently without blocking or waiting for each task to complete.
2. Non-Blocking I/O: CrewAI asynchronous tools provide non-blocking I/O operations, allowing developers to perform input/output operations without blocking the execution of other tasks.
3. Concurrent Execution: CrewAI asynchronous tools enable concurrent execution of tasks, improving system performance and responsiveness by utilizing multiple CPU cores and minimizing idle time.
4. Task Queuing: CrewAI

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 56b49adb-614f-4b0b-abec-a0a31ad4bc64                                                                       │
│  Final Output: CrewAI asynchronous tools are designed to streamline workflows and enhance productivity by       │
│  allowing for non-blocking, concurrent execution of tasks. These tools enable developers to write               │
│  single-threaded code that can handle multiple tasks simultaneously, improving overall system performance and   │
│  responsiveness.                                                                                                │
│                                                                                                                 │
│  Key features of CrewAI asynchronous tools include:                                                             │
│                                                                                                                 │
│  1. Asynchronous Programming: CrewAI asynchronous tools support asynchronous programming models, enabling       │
│  developers to write code that can execute multiple tasks concurrently without blocking or waiting for each     │
│  task to complete.                                                                                              │
│  2. Non-Blocking I/O: CrewAI asynchronous tools provide non-blocking I/O operations, allowing developers to     │
│  perform input/output operations without blocking the execution of other tasks.                                 │
│  3. Concurrent Execution: CrewAI asynchronous tools enable concurrent execution of tasks, improving system      │
│  performance and responsiveness by utilizing multiple CPU cores and minimizing idle time.                       │
│  4. Task Queuing: CrewAI asynchronous tools often include task queuing mechanisms, which allow developers to    │
│  schedule tasks for execution and manage task priorities.                                                       │
│  5. Error Handling: CrewAI asynchronous tools typically provide robust error handling mechanisms, enabling      │
│  developers to handle errors and exceptions that may occur during asynchronous task execution.                  │
│                                                                                                                 │
│  By leveraging CrewAI asynchronous tools, developers can build scalable, high-performance applications that     │
│  can handle large volumes of concurrent requests and tasks, improving overall system efficiency and user        │
│  experience.                                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# Alternative approach.....

## 2. Implementing Async Methods in Custom Tool Classes

In [22]:
# -----------------------------------------------------
# Define the custom async tool
# -----------------------------------------------------

from crewai.tools import BaseTool
import asyncio

class AsyncCustomTool(BaseTool):
    name: str = "async_custom_tool"
    description: str = "An asynchronous custom tool"

    async def _run(self, query: str = "") -> str:
        """Asynchronously run the tool"""
        # Your async implementation here
        await asyncio.sleep(1)
        return f"Processed {query} asynchronously"

In [34]:
# -----------------------------------------------------
# Define the agent
# -----------------------------------------------------
research_agent = Agent(
    role="Research Assistant",
    goal="Retrieve information using the available tool and summarize it.",
    backstory=(
        "You are an AI research assistant that can fetch external "
        "information using custom tools."
    ),
    tools=[AsyncCustomTool()],

    llm=llm,
    verbose=True
)

# -----------------------------------------------------
# Define the task
# -----------------------------------------------------

research_task = Task(
    description=(
        "Find information about CrewAI asynchronous tools "
        "by using the available tool. "
        "Summarize the retrieved information."
    ),
    expected_output="A short summary of the retrieved data.",
    agent=research_agent
)

# -----------------------------------------------------
# Define the crew
# -----------------------------------------------------

crew = Crew(
    agents=[research_agent],
    tasks=[research_task],
    process=Process.sequential,
    verbose=True
)


# -----------------------------------------------------
# Execute
# -----------------------------------------------------

result = await crew.kickoff_async()

print(result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 5a014dff-1771-4701-ae26-173f0072f98e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Find information about CrewAI asynchronous tools by using the available tool. Summarize the retrieved    │
│  information.                                                                                                   │
│  ID: 60aa87d8-8d21-426c-bbb8-b622a2482e5e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Assistant                                                                                      │
│                                                                                                                 │
│  Task: Find information about CrewAI asynchronous tools by using the available tool. Summarize the retrieved    │
│  information.                                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: async_custom_tool                                                                                        │
│  Args: {'query': 'CrewAI asynchronous tools'}                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool async_custom_tool executed with result: Processed CrewAI asynchronous tools asynchronously...


...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Assistant                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  CrewAI asynchronous tools are designed to streamline workflows and enhance productivity. These tools enable    │
│  users to perform tasks in the background, allowing for uninterrupted work on other projects. With CrewAI's     │
│  asynchronous capabilities, teams can automate repetitive tasks, manage complex workflows, and collaborate      │
│  more efficiently. The tools provide real-time updates and notifications, ensuring that all team members are    │
│  informed and aligned. By leveraging CrewAI's asynchronous tools, organizations can improve their overall       │
│  efficiency, reduce manual errors, and increase output.                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Find information about CrewAI asynchronous tools by using the available tool. Summarize the retrieved    │
│  information.                                                                                                   │
│  Agent: Research Assistant                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

CrewAI asynchronous tools are designed to streamline workflows and enhance productivity. These tools enable users to perform tasks in the background, allowing for uninterrupted work on other projects. With CrewAI's asynchronous capabilities, teams can automate repetitive tasks, manage complex workflows, and collaborate more efficiently. The tools provide real-time updates and notifications, ensuring that all team members are informed and aligned. By leveraging CrewAI's asynchronous tools, organizations can improve their overall efficiency, reduce manual errors, and increase output.


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 56b49adb-614f-4b0b-abec-a0a31ad4bc64                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Find information about CrewAI asynchronous tools by using the available tool. Summarize the retrieved    │
│  information.                                                                                                   │
│  ID: 77d83b26-b18a-4bc6-a5cf-384c8c729748                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Assistant                                                                                      │
│                                                                                                                 │
│  Task: Find information about CrewAI asynchronous tools by using the available tool. Summarize the retrieved    │
│  information.                                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#3) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: fetch_data_async                                                                                         │
│  Args: {'query': 'CrewAI asynchronous tools'}                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool fetch_data_async executed with result: Data retrieved for CrewAI asynchronous tools...


...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Assistant                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  CrewAI asynchronous tools are designed to streamline workflows and enhance productivity by enabling teams to   │
│  work efficiently in the background. These tools include features such as automated task assignment, real-time  │
│  notifications, and customizable workflows, allowing teams to manage complex projects with ease. Additionally,  │
│  CrewAI's asynchronous tools provide advanced analytics and reporting capabilities, providing valuable          │
│  insights into team performance and project progress. With CrewAI's asynchronous tools, teams can collaborate   │
│  more effectively, reduce errors, and increase overall productivity.                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Find information about CrewAI asynchronous tools by using the available tool. Summarize the retrieved    │
│  information.                                                                                                   │
│  Agent: Research Assistant                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

CrewAI asynchronous tools are designed to streamline workflows and enhance productivity by enabling teams to work efficiently in the background. These tools include features such as automated task assignment, real-time notifications, and customizable workflows, allowing teams to manage complex projects with ease. Additionally, CrewAI's asynchronous tools provide advanced analytics and reporting capabilities, providing valuable insights into team performance and project progress. With CrewAI's asynchronous tools, teams can collaborate more effectively, reduce errors, and increase overall productivity.


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# **Tool with Caching**

Caching is an important optimization feature in CrewAI that prevents unnecessary repeated execution of tools.

Many tools perform expensive operations such as calling external APIs, querying databases, searching the web, or running computationally intensive algorithms. Without caching, if the same tool is invoked multiple times with the same inputs, the operation is repeated each time, increasing execution time, API costs, and system load.

With caching enabled, CrewAI stores the result of a tool invocation and returns the cached result for subsequent identical requests, eliminating redundant computation.
CrewAI also allows developers to define custom caching policies through a cache_function, enabling selective caching based on the tool's input arguments or output.
This flexibility allows frequently reused or deterministic results to be cached while avoiding caching for dynamic or time-sensitive operations, thereby improving both performance and resource efficiency.

In [105]:
from crewai.tools import tool
import time


# -------------------------------------------------------
# Custom Tool
# -------------------------------------------------------
@tool
def multiplication_tool(first_number: int, second_number: int) -> int:
    """
    Multiply two numbers.
    """

    print(">>> Tool is actually executing...")
    time.sleep(2)            # Simulate an expensive computation

    return first_number * second_number


# -------------------------------------------------------
# Cache Policy
# -------------------------------------------------------
# def cache_func(args, result):
#     """
#     Cache only EVEN results.
#     """
#     print(f"Cache decision for {result}")
#     return result % 2 == 0

# OR
def cache_func(args, result):
    print(f"Cache decision for {result}")
    return result > 100


multiplication_tool.cache_function = cache_func


In [86]:
# print(multiplication_tool)
# print()
# print(multiplication_tool.cache_function)
# print()
# print(hasattr(multiplication_tool, "cache_function"))

In [106]:
from crewai import Agent, Task, Crew, Process, LLM

# -----------------------------------------------------
# Configure the LLM
# -----------------------------------------------------
# GROQ hosted LLMs
llm = LLM(
     model="llama-3.3-70b-versatile",
     base_url="https://api.groq.com/openai/v1",
     api_key=os.environ["GROQ_API_KEY"],
     temperature=0.4)


# -------------------------------------------------------
# Agent
# -------------------------------------------------------

math_agent = Agent(
    role="Math Teacher",
    goal="Help students perform multiplication.",
    backstory="You are a math teacher.",
    llm=llm,
    tools=[multiplication_tool],
    cache=True, # <= caching enabled
    verbose=True
)


# -------------------------------------------------------
# Task
# -------------------------------------------------------

task = Task(
    description="""
Use the multiplication tool to calculate the results of multiplying two numbers based on the user {query}.
Return only the answer.
""",
    expected_output="Result of multiplication of two numbers",
    agent=math_agent,

)


# -------------------------------------------------------
# Crew
# -------------------------------------------------------

crew = Crew(
    agents=[math_agent],
    tasks=[task],
    process=Process.sequential
)


In [88]:
# print(math_agent.tools)
# print(math_agent.tools[0])
# print(math_agent.tools[0].cache_function)

In [107]:
# -------------------------------------------------------
# First execution
# -------------------------------------------------------

print("\nFIRST RUN\n")

result = await crew.kickoff_async(
    inputs={
        "query": "5 times 3"
    }
)

print(result.raw)



FIRST RUN



╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Math Teacher                                                                                            │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Use the multiplication tool to calculate the results of multiplying two numbers based on the user 5 times 3.   │
│  Return only the answer.                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

>>> Tool is actually executing...
Cache decision for 15
Tool multiplication_tool executed with result: 15...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Math Teacher                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Result of multiplication of two numbers: 15                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Result of multiplication of two numbers: 15


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [108]:
# -------------------------------------------------------
# Second execution
# -------------------------------------------------------

print("\nSECOND RUN\n")

result = await crew.kickoff_async(
    inputs={
        "query": "20 times 6"
    }
)

print(result.raw)



SECOND RUN



╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Math Teacher                                                                                            │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Use the multiplication tool to calculate the results of multiplying two numbers based on the user 20 times 6.  │
│  Return only the answer.                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

>>> Tool is actually executing...
Cache decision for 120
Tool multiplication_tool executed with result: 120...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Math Teacher                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The result of multiplying 20 and 6 is 120.                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

The result of multiplying 20 and 6 is 120.


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [109]:
# -------------------------------------------------------
# Third execution with identical arguments (Triggers Cache)
# -------------------------------------------------------

print("\nTHIRD RUN\n")

result = await crew.kickoff_async(
    inputs={
        "query": "20 times 6"
    }
)

print(result.raw)



THIRD RUN



╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Math Teacher                                                                                            │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Use the multiplication tool to calculate the results of multiplying two numbers based on the user 20 times 6.  │
│  Return only the answer.                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool multiplication_tool executed with result (from cache): 120...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Math Teacher                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The result of multiplying 20 and 6 is 120.                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

The result of multiplying 20 and 6 is 120.


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# **Tool Failure Policy**

A failure policy transforms low-level exceptions into LLM-friendly feedback.

Instead of receiving an opaque Python traceback, the agent receives a clear explanation of what went wrong. This improves robustness in real-world applications where tools depend on external APIs, databases, file systems, or network services that may occasionally fail.
It also enables the agent to recover gracefully by retrying, choosing an alternative tool, requesting additional user input, or explaining the failure to the user in a meaningful way.
This aligns with CrewAI's emphasis on building production-ready agents that can handle tool failures gracefully rather than simply crashing.

## Create a tool that may fail

In [124]:
from crewai.tools import tool
import random

@tool("weather_tool")
def weather_tool(city: str) -> str:
    """Returns the current weather for a city."""

    # Simulate an unreliable API
    if random.random() < 0.5:
        raise ConnectionError("Weather API is temporarily unavailable.")

    return f"The current temperature in {city} is 31°C."

## Define a custom failure policy

In [145]:
# Alternatively select the tool failure policy and execute the crew

# Soft
# Nothing is recorded, emitted, or acted on.
# weather_tool.tool_failure_policy = "ignore" # failure is treated as non-critical and doesn't generate a warning.


# Mild
# Records the failure, emits ToolFailureDetectedEvent, and continues.
weather_tool.tool_failure_policy = "warn" # DEFAULT, The user still gets partial results.


# Restrictive
# Records and emits, then aborts with ToolExecutionFailedError
# weather_tool.tool_failure_policy = "raise" # Nothing is returned.


## Create an agent

In [132]:
from crewai import Agent

weather_agent = Agent(
    role="Weather Assistant",
    goal="Answer weather-related questions.",
    backstory="You help users with weather information.",
    tools=[weather_tool],
    verbose=True
)

# Create Task

In [146]:
from crewai import Task

weather_task = Task(
    description="""
Find the current weather in Delhi.
Use the weather tool.
""",
    expected_output="Current weather conditions.",
    agent=weather_agent
)

# Create Crew

In [147]:
from crewai import Crew

crew = Crew(
    agents=[weather_agent],
    tasks=[weather_task],
    verbose=True
)

# Run the Crew

In [148]:
# Run this cell multiple times

result = await crew.kickoff_async()

print(result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 76572cfd-87ff-4771-b183-4189c60ad9ee                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  Find the current weather in Delhi.                                                                             │
│  Use the weather tool.                                                                                          │
│                                                                                                                 │
│  ID: 95ed0659-a1e7-4bbf-96b6-e1ce6a25a5fe                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Weather Assistant                                                                                       │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Find the current weather in Delhi.                                                                             │
│  Use the weather tool.                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool weather_tool executed with result: Error executing tool: Weather API is temporarily unavailable....


╭──────────────────────────────────────── 🔧 Tool Execution Started (#13) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: weather_tool                                                                                             │
│  Args: {'city': 'Delhi'}                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Weather Assistant                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  I'm sorry, but the weather service is temporarily unavailable right now, so I cannot provide the current       │
│  weather in Delhi at the moment. Please try again later.                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Find the current weather in Delhi.                                                                             │
│  Use the weather tool.                                                                                          │
│                                                                                                                 │
│  Agent: Weather Assistant                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

I'm sorry, but the weather service is temporarily unavailable right now, so I cannot provide the current weather in Delhi at the moment. Please try again later.


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯